In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, SimpleRNN, LSTM, Dense

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

X = np.random.randn(500, 20, 1).astype(np.float32)
y = (X[:, 0, 0] > 0).astype(np.float32).reshape(-1, 1)

split_idx = int(0.8 * len(X))
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

In [ ]:
def build_rnn_model():
    inputs = Input(shape=(20, 1))
    rnn = SimpleRNN(32)(inputs)
    outputs = Dense(1, activation='sigmoid')(rnn)
    model = Model(inputs, outputs)
    model.compile(optimizer='sgd', loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_lstm_model():
    inputs = Input(shape=(20, 1))
    lstm = LSTM(32)(inputs)
    outputs = Dense(1, activation='sigmoid')(lstm)
    model = Model(inputs, outputs)
    model.compile(optimizer='sgd', loss='binary_crossentropy', metrics=['accuracy'])
    return model

rnn_model = build_rnn_model()
lstm_model = build_lstm_model()

rnn_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50, batch_size=32, verbose=0)
lstm_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50, batch_size=32, verbose=0)

In [ ]:
X_batch = tf.convert_to_tensor(X_val)
y_batch = tf.convert_to_tensor(y_val)

with tf.GradientTape() as tape:
    tape.watch(X_batch)
    predictions_rnn = rnn_model(X_batch)
    loss_rnn = tf.keras.losses.binary_crossentropy(y_batch, predictions_rnn)
    mean_loss_rnn = tf.reduce_mean(loss_rnn)

grads_rnn = tape.gradient(mean_loss_rnn, X_batch)
mean_abs_grads_rnn = tf.reduce_mean(tf.abs(grads_rnn), axis=[0, 2]).numpy()

with tf.GradientTape() as tape:
    tape.watch(X_batch)
    predictions_lstm = lstm_model(X_batch)
    loss_lstm = tf.keras.losses.binary_crossentropy(y_batch, predictions_lstm)
    mean_loss_lstm = tf.reduce_mean(loss_lstm)

grads_lstm = tape.gradient(mean_loss_lstm, X_batch)
mean_abs_grads_lstm = tf.reduce_mean(tf.abs(grads_lstm), axis=[0, 2]).numpy()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(mean_abs_grads_rnn, label='SimpleRNN', color='red', marker='o')
plt.plot(mean_abs_grads_lstm, label='LSTM', color='blue', marker='s')
plt.title('Mean Absolute Gradient w.r.t Input at Each Time Step')
plt.xlabel('Time Step')
plt.ylabel('Mean Absolute Gradient')
plt.xticks(range(20))
plt.grid(True)
plt.legend()
plt.show()

### Analysis

1. **For the RNN, how does gradient magnitude at step 0 compare to step 19?**
   - In the SimpleRNN model, the gradient magnitude at time step 0 is extremely small (virtually zero) compared to step 19. This is the vanishing gradient problem in action; as the gradient backpropagates from the final step (19) back to the first step (0), the repeated matrix multiplications with weights whose eigenvalues are less than 1 cause the gradient to decay exponentially.

2. **For the LSTM, is the gradient at step 0 relatively larger? Why?**
   - Yes, the gradient at step 0 is significantly larger in the LSTM than in the RNN.
   - This occurs because of the LSTM's additive cell state update path (controlled by the input and forget gates), which creates a linear highway for gradient flow. Instead of solely multiplicative decay through recurrent weights, gradients can flow backwards through the cell state with minimal attenuation.

3. **How does this explain why LSTM performs better on long-range dependency tasks?**
   - Since LSTMs preserve non-vanishing gradient signals back to the very first steps of the sequence, the model can update weights at early time steps based on errors computed at the end of the sequence. This enables learning relationships over long spans of time steps (such as step 0 affecting step 19), whereas the SimpleRNN receives no meaningful learning signal at step 0.